# Qwirkle AlphaZero Training on Kaggle GPU

**Pipeline** (~8h total on T4):
1. Setup Rust (~2 min)
2. Clone & build (~15 min)
3. Self-play 1000 games (~5 min)
4. Train large model (~30 min)
5. AlphaZero loop 15 iters (~6h)
6. Evaluate

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('PyTorch:', torch.__version__)

TORCH_LIB = os.path.join(os.path.dirname(torch.__file__), 'lib')
os.environ['LIBTORCH_USE_PYTORCH'] = '1'
os.environ['LIBTORCH_BYPASS_VERSION_CHECK'] = '1'
os.environ['LD_LIBRARY_PATH'] = f'{TORCH_LIB}:' + os.environ.get('LD_LIBRARY_PATH', '')
%env TORCH_LIB={TORCH_LIB}
%env LD_LIBRARY_PATH={TORCH_LIB}
%env LIBTORCH_USE_PYTORCH=1
%env LIBTORCH_BYPASS_VERSION_CHECK=1
print('TORCH_LIB:', TORCH_LIB)

In [ ]:
%%bash
# Install Rust (if needed)
if ! command -v $HOME/.cargo/bin/rustc &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
fi
source $HOME/.cargo/env
rustc --version

In [ ]:
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ['PATH']

In [ ]:
%%bash
cd /kaggle/working
rm -rf qwirkle
git clone --depth 1 -b dev https://github.com/specialjcg/qwirkle.git
echo 'Cloned.'

In [ ]:
%%bash
source $HOME/.cargo/env
export LIBTORCH_USE_PYTORCH=1
export LIBTORCH_BYPASS_VERSION_CHECK=1
cd /kaggle/working/qwirkle/backend
echo 'Building... (this takes ~15 min on first run)'
cargo build --features neural --release 2>&1 | tail -10
echo ''
echo 'Binaries:'
ls -lh target/release/{selfplay,train_bot,alphazero,evaluate} 2>&1

## Self-play data generation (1000 games, ~5 min)

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
mkdir -p /kaggle/working/data /kaggle/working/models
./target/release/selfplay \
    --games 1000 \
    --out /kaggle/working/data/bootstrap.bin \
    --epsilon 0.1
ls -lh /kaggle/working/data/

## Train large model (~30 min)

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/train_bot \
    --data /kaggle/working/data/bootstrap.bin \
    --epochs 60 \
    --batch 256 \
    --lr 0.001 \
    --out /kaggle/working/models/v6_large.pt \
    --patience 15 \
    --policy-weight 0.5 \
    --large
ls -lh /kaggle/working/models/

## Eval base model vs Greedy

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/evaluate \
    --model /kaggle/working/models/v6_large.pt \
    --games 50 \
    --large

## AlphaZero loop (~6h)

15 iterations, mixed self-play (50% vs greedy), arena vs greedy.
Target: 75% vs greedy.

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/alphazero \
    --iterations 15 \
    --selfplay-games 50 \
    --mcts-sims 80 \
    --arena-games 30 \
    --eval-every 2 \
    --eval-games 30 \
    --target-greedy-winrate 0.75 \
    --win-threshold 0.55 \
    --out-dir /kaggle/working/models/az_best \
    --init-model /kaggle/working/models/v6_large.pt \
    --large

## Final evaluation

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend

MODEL=/kaggle/working/models/az_best/best.pt
if [ ! -f "$MODEL" ]; then
    MODEL=/kaggle/working/models/v6_large.pt
fi

echo '=== Value-only ==='
./target/release/evaluate --model $MODEL --games 100 --large

echo ''
echo '=== With MCTS 80 ==='
./target/release/evaluate --model $MODEL --games 50 --mcts 80 --large

In [ ]:
%%bash
# Cleanup heavy files, keep only models
rm -f /kaggle/working/data/bootstrap.bin
find /kaggle/working/models/az_best -name 'iter*_samples.bin' -delete 2>/dev/null
find /kaggle/working/models/az_best -name 'iter*_candidate.pt' -delete 2>/dev/null
echo '=== Final output ==='
ls -lhR /kaggle/working/models/ 2>/dev/null
du -sh /kaggle/working/models/